### Imports

In [ ]:
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import time

# Import your custom modules
from dynaphos import utils, cortex_models
from dynaphos.simulator import GaussianSimulator

print("Dependencies loaded successfully!")

### Configuration and parameter setup

In [ ]:
# Load the simulator configuration file
params = utils.load_params('../config/params.yaml')

# Disable temporal dynamics and thresholding for static image demonstration
params['temporal_dynamics']['trace_increase_rate'] = 0.0
params['temporal_dynamics']['trace_decay_per_second'] = 0.9999999
params['temporal_dynamics']['activation_decay_per_second'] = 0.999999
params['thresholding']['rheobase'] = 0.0

print("Parameters configured!")

### Initialize simulators with the different raster patterns

In [ ]:
n_phosphenes = 100
phosphene_coords = cortex_models.get_visual_field_coordinates_probabilistically(params, n_phosphenes)

# Create simulators for each raster pattern
raster_patterns = ['horizontal', 'vertical', 'checkerboard', 'random']
simulators = {}

for pattern in raster_patterns:
    simulators[pattern] = GaussianSimulator(
        params, 
        phosphene_coords,
        raster_enabled=True,
        raster_pattern=pattern,
        raster_num_groups=5,
        raster_rate_hz=4.5
    )

# Also create one without raster for comparison
simulators['no_raster'] = GaussianSimulator(
    params, 
    phosphene_coords,
    raster_enabled=False
)

print(f"Initialized {len(simulators)} simulators:")
for pattern in simulators.keys():
    print(f"  - {pattern}")

### Load and prepare video

In [ ]:
# Load the videostream
cap = cv2.VideoCapture('./gradient_static.mp4')
if not cap.isOpened():
    # Try alternative path or use a test image
    print('Unable to read video file, using test pattern instead')
    # Create a test gradient pattern
    test_frame = np.zeros((256, 256), dtype=np.uint8)
    for i in range(256):
        test_frame[:, i] = i
    use_test_pattern = True
else:
    use_test_pattern = False

framerate = params['run']['fps']
max_n_frames = min(60 * framerate, 100)  # Limit for demonstration

print(f"Video loaded: {not use_test_pattern}")
print(f"Max frames to process: {max_n_frames}")

### Single Frame test

In [ ]:
# Cell 6: Single frame test with all patterns
# Process one frame to show all patterns
if use_test_pattern:
    frame = test_frame
else:
    ret, frame = cap.read()
    if ret:
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        # Get square if frame is not square
        if frame.shape[0] != frame.shape[1]:
            shortest_side = min(frame.shape)
            frame = frame[frame.shape[0]//2-shortest_side//2:frame.shape[0]//2+shortest_side//2,
                          frame.shape[1]//2-shortest_side//2:frame.shape[1]//2+shortest_side//2]

# Preprocess frame
frame = cv2.resize(frame, (256, 256))
frame = cv2.GaussianBlur(frame, (21, 21), 5)
processed_img = frame

# Process with each simulator
results = {}
charge_data = {}

for pattern, simulator in simulators.items():
    stim_pattern = simulator.sample_stimulus(processed_img, rescale=True)
    phs = simulator(stim_pattern).clamp(0, 1)
    phs = utils.to_numpy(phs) * 255
    results[pattern] = phs
    
    charge_status = simulator.get_charge_status()
    charge_data[pattern] = charge_status

# Display results
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

patterns_to_show = ['no_raster'] + raster_patterns
for idx, pattern in enumerate(patterns_to_show):
    if idx < len(axes):
        ax = axes[idx]
        if pattern in results:
            # Show original + phosphene result
            combined = np.concatenate([processed_img, results[pattern]], axis=1)
            ax.imshow(combined, cmap='gray', vmin=0, vmax=255)
            ax.set_title(f'{pattern.upper()}\nMax Charge: {charge_data[pattern]["max_charge_uC"]:.1f}µC')
            ax.axis('off')

# Hide unused subplots
for idx in range(len(patterns_to_show), len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.show()

### Interactive comparison over multiple frames

In [ ]:
print("Starting interactive comparison...")
print("Press 'q' in the OpenCV window to stop early")

# Reset all simulators
for simulator in simulators.values():
    simulator.reset()

# Create a simple interactive display
frame_nr = 0
performance_data = {pattern: [] for pattern in simulators.keys()}

while frame_nr < max_n_frames:
    if use_test_pattern:
        frame = test_frame.copy()
        # Modify test pattern slightly to see temporal effects
        frame = (frame + frame_nr) % 256
    else:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        if frame.shape[0] != frame.shape[1]:
            shortest_side = min(frame.shape)
            frame = frame[frame.shape[0]//2-shortest_side//2:frame.shape[0]//2+shortest_side//2,
                          frame.shape[1]//2-shortest_side//2:frame.shape[1]//2+shortest_side//2]
    
    # Preprocess
    frame = cv2.resize(frame, (256, 256))
    frame = cv2.GaussianBlur(frame, (21, 21), 5)
    processed_img = frame
    
    # Process with all simulators
    results = {}
    for pattern, simulator in simulators.items():
        stim_pattern = simulator.sample_stimulus(processed_img, rescale=True)
        phs = simulator(stim_pattern).clamp(0, 1)
        phs = utils.to_numpy(phs) * 255
        results[pattern] = phs
        
        # Collect performance data every 10 frames
        if frame_nr % 10 == 0:
            charge_status = simulator.get_charge_status()
            performance_data[pattern].append({
                'frame': frame_nr,
                'max_charge': charge_status['max_charge_uC'],
                'max_electrode': charge_status['max_electrode_idx']
            })
    
    # Create comparison display
    comparison_rows = []
    current_row = []
    
    for i, pattern in enumerate(['no_raster'] + raster_patterns):
        if pattern in results:
            combined = np.concatenate([processed_img, results[pattern]], axis=1).astype(np.uint8)
            
            # Add text label
            combined_display = cv2.cvtColor(combined, cv2.COLOR_GRAY2BGR)
            cv2.putText(combined_display, pattern, (10, 30), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            
            current_row.append(combined_display)
            
            if len(current_row) == 2:
                comparison_rows.append(np.concatenate(current_row, axis=1))
                current_row = []
    
    if current_row:
        if len(current_row) == 1:
            # Pad with black image
            blank = np.zeros_like(current_row[0])
            current_row.append(blank)
        comparison_rows.append(np.concatenate(current_row, axis=1))
    
    final_comparison = np.concatenate(comparison_rows, axis=0)
    
    # Display in notebook
    clear_output(wait=True)
    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(final_comparison, cv2.COLOR_BGR2RGB))
    plt.title(f'Frame {frame_nr} - Raster Pattern Comparison\n(Press q in OpenCV window to stop)')
    plt.axis('off')
    plt.show()
    
    # Also show charge status
    print(f"\nFrame {frame_nr} Charge Status:")
    print("-" * 50)
    for pattern in ['no_raster'] + raster_patterns:
        charge_status = simulators[pattern].get_charge_status()
        raster_enabled = hasattr(simulators[pattern], 'raster_enabled') and simulators[pattern].raster_enabled
        current_group = simulators[pattern].current_raster_group if raster_enabled else 'N/A'
        
        print(f"{pattern:12} | Charge: {charge_status['max_charge_uC']:6.1f} µC | "
              f"Electrode: {charge_status['max_electrode_idx']:3d} | Group: {current_group}")
    
    frame_nr += 1
    
    # Check for early stop
    if cv2.waitKey(1) & 0xFF == ord('q'):
        print("\nStopped early by user")
        break
        
    # Small delay to make animation visible
    time.sleep(0.1)

cv2.destroyAllWindows()
print(f"\nProcessed {frame_nr} frames")

### Performance analysis and visualization

In [ ]:
print("Performance Analysis")
print("=" * 60)

# Create charge accumulation plots
plt.figure(figsize=(12, 8))

for pattern in simulators.keys():
    if performance_data[pattern]:
        frames = [data['frame'] for data in performance_data[pattern]]
        charges = [data['max_charge'] for data in performance_data[pattern]]
        plt.plot(frames, charges, 'o-', label=pattern, linewidth=2, markersize=4)

plt.axhline(y=params.get('safety', {}).get('cumulative_charge_limit_uC', 30.0), 
           color='r', linestyle='--', label='Safety Limit')
plt.xlabel('Frame Number')
plt.ylabel('Max Cumulative Charge (µC)')
plt.title('Charge Accumulation Comparison Across Raster Patterns')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Final statistics
print("\nFINAL STATISTICS:")
print("-" * 50)
for pattern in ['no_raster'] + raster_patterns:
    charge_status = simulators[pattern].get_charge_status()
    final_charge = charge_status['max_charge_uC']
    safety_limit = params.get('safety', {}).get('cumulative_charge_limit_uC', 30.0)
    
    status = "SAFE" if final_charge <= safety_limit else "WARNING"
    
    print(f"{pattern:12} | Final Charge: {final_charge:6.1f} µC | {status} | "
          f"Electrode: {charge_status['max_electrode_idx']}")

### Raster pattern efficiency analysis

In [ ]:
print("Raster Pattern Efficiency Analysis")
print("=" * 50)

for pattern in raster_patterns:
    simulator = simulators[pattern]
    if hasattr(simulator, 'raster_enabled') and simulator.raster_enabled:
        raster_info = simulator.get_raster_info()
        electrodes_per_group = raster_info.get('electrodes_per_group', [])
        
        if electrodes_per_group:
            imbalance = max(electrodes_per_group) - min(electrodes_per_group)
            print(f"\n{pattern.upper():12}")
            print(f"  Groups: {raster_info['num_groups']}")
            print(f"  Electrodes per group: {electrodes_per_group}")
            print(f"  Load imbalance: {imbalance} electrodes")
            print(f"  Raster rate: {raster_info['raster_rate_hz']} Hz")
            print(f"  Group interval: {raster_info['group_interval_s']:.3f} s")

# Calculate and display key metrics
print("\nKEY METRICS:")
print("-" * 30)
baseline_charge = performance_data['no_raster'][-1]['max_charge'] if performance_data['no_raster'] else 0

for pattern in raster_patterns:
    if performance_data[pattern]:
        final_charge = performance_data[pattern][-1]['max_charge']
        if baseline_charge > 0:
            reduction = ((baseline_charge - final_charge) / baseline_charge) * 100
            print(f"{pattern:12}: {reduction:+.1f}% charge reduction vs no raster")

### Cleanup and final thoughts

In [ ]:
# Release video capture if used
if not use_test_pattern:
    cap.release()
cv2.destroyAllWindows()

print("Test completed! 🎉")
print("\nSummary of findings:")
print("• Compare the visual quality in the interactive display")
print("• Check the charge accumulation plots for safety performance")
print("• Review the efficiency metrics for each pattern")
print("• Consider both visual quality and safety for your application")

print(f"\nGenerated data for {frame_nr} frames across {len(simulators)} configurations")